## Exercise 14 

In this problem, you will develop a model to predict whether a given car gets 
high or low gas mileage based on the `Auto` data set.

**(a)** Create a binary variable `mpg01` = 1 if `mpg` is above its median, 0 otherwise:

$$\text{mpg01} = \begin{cases} 1 & \text{if } \text{mpg} > \text{median(mpg)} \\ 0 & \text{if } \text{mpg} < \text{median(mpg)} \end{cases}$$

**(b)** Explore the data graphically to investigate the association between `mpg01` 
and the other features. Which features seem most useful for predicting `mpg01`?

**(c)** Split the data into a training set and a test set.

**(d)** Perform LDA on the training data to predict `mpg01` using the variables 
selected in (b). What is the test error?

**(e)** Repeat (d) using QDA.

**(f)** Repeat (d) using logistic regression.

**(g)** Repeat (d) using naive Bayes.

**(h)** Perform KNN with several values of $K$ to predict `mpg01`. 
Which value of $K$ performs best?

In [1]:
import numpy as np
import pandas as pd
from matplotlib .pyplot import subplots
import statsmodels .api as sm
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS ,
summarize )

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
( LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn. naive_bayes import GaussianNB
from sklearn. neighbors import KNeighborsClassifier
from sklearn. preprocessing import StandardScaler
from sklearn. model_selection import train_test_split
from sklearn. linear_model import LogisticRegression

import seaborn as sns 
import matplotlib.pyplot as plt 


In [2]:

Auto = pd.read_csv("../datasets/Auto.data", na_values=["?"], delim_whitespace=True)
Auto.head()
Auto = Auto.dropna()

/var/folders/gq/kz8zkjwj6k1dr27f7byh1x_w0000gn/T/ipykernel_2079/1763140999.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  Auto = pd.read_csv("../datasets/Auto.data", na_values=["?"], delim_whitespace=True)


In [3]:

mpg_median = Auto['mpg'].median()
mpg01 = np.array([0]*Auto.mpg.shape[0])
mpg01[Auto.mpg > mpg_median] = 1
Auto['mpg01'] = mpg01



In [ ]:
numeric_col = Auto.columns.drop('name')
corr = Auto[numeric_col].corr()

fig,ax = subplots(figsize=(10,6))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Auto - Correlation Heatmap',fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:

features = Auto.columns.drop(['mpg01','mpg','name'])
fig,ax = plt.subplots(2,4,figsize=(16,8))
ax = ax.flatten()

for i,feat in enumerate(features):
    sns.boxplot(x=Auto['mpg01'],y=Auto[feat],ax=ax[i])
    ax[i].set_title(feat)

ax[-1].set_visible(False)
plt.tight_layout()
plt.show()

We can see the separation of the distributions of different predictors (y-axis) across the two classes of mpg01. The more separated the distributions, the more relevant that predictor is for determining whether mpg01 is 0 or 1.

In [ ]:
pd.plotting.scatter_matrix(Auto, figsize=(12,12));

The scatter matrix reveals that many predictors influence mpg but are also strongly correlated with each other — this is a case of multicollinearity. Including all of them in the model may cause problems.

## LDA for mpg01 prediction:

In [ ]:
Auto.describe()

In [14]:
Data_train,Data_test =  Auto.iloc[:300],Auto.iloc[300:]
variables = Auto.columns.drop(['mpg','mpg01','name','acceleration','year'])
design = MS(variables)

X_train = design.fit_transform(Data_train).drop(columns=['intercept'])
X_test = design.transform(Data_test).drop(columns=['intercept'])
y_train = Data_train['mpg01'].values
y_test = Data_test['mpg01'].values

lda = LDA(store_covariance=True)
results_lda = lda.fit(X_train,y_train) 

pred_lda = results_lda.predict(X_test)

print(round(float(np.mean(pred_lda != y_test)),3))
confusion_table(pred_lda,y_test)

0.109


Truth,0,1
Predicted,,
0,5,10
1,0,77


### QDA 

In [13]:
Auto_s = Auto.sample(frac=1, random_state=42).reset_index(drop=True)
Data_train, Data_test = Auto_s.iloc[:300], Auto_s.iloc[300:]

variables = Auto.columns.drop(['mpg','mpg01','name','acceleration','year'])
design = MS(variables)

X_train = design.fit_transform(Data_train).drop(columns=['intercept'])
X_test = design.transform(Data_test).drop(columns=['intercept'])
y_train = Data_train['mpg01'].values
y_test = Data_test['mpg01'].values

qda = QDA(store_covariance=True,reg_param = 0.01)
results_qda = qda.fit(X_train,y_train) 

pred_qda = results_qda.predict(X_test)

print("model accuracy error: ",round(float(np.mean(pred_qda != y_test)),3))
confusion_table(pred_qda,y_test)

model accuracy error:  0.065


Truth,0,1
Predicted,,
0,36,5
1,1,50


### Logistic Regression 

In [16]:
Auto_s = Auto.sample(frac=1, random_state=42).reset_index(drop=True)
Data_train, Data_test = Auto_s.iloc[:300], Auto_s.iloc[300:]
variables = Auto.columns.drop(['mpg','mpg01','name','acceleration','year'])
design = MS(variables)

X_train = design.fit_transform(Data_train)
X_test = design.transform(Data_test)
y_train = Data_train['mpg01'].values
y_test = Data_test['mpg01'].values

logit_train = sm.GLM(y_train,X_train,family=sm.families.Binomial())
results_logit = logit_train.fit()
print(summarize(results_logit))

pred_logit = results_logit.predict(X_test)
pred_logit[pred_logit > 0.5] = 1.
pred_logit[pred_logit < 0.5] = 0.

print("model accuracy error: ",round(float(np.mean(pred_logit != y_test)),3))
confusion_table(pred_logit,y_test)

                 coef  std err      z  P>|z|
intercept     11.4918    2.130  5.396  0.000
cylinders     -0.0194    0.412 -0.047  0.962
displacement  -0.0154    0.011 -1.421  0.155
horsepower    -0.0503    0.017 -3.025  0.002
weight        -0.0015    0.001 -1.802  0.072
origin        -0.0272    0.313 -0.087  0.931
model accuracy error:  0.098


Truth,0,1
Predicted,,
0,35,7
1,2,48


### Naive Bayes

In [5]:
Auto_s = Auto.sample(frac=1, random_state=42).reset_index(drop=True)
Data_train, Data_test = Auto_s.iloc[:300], Auto_s.iloc[300:]

variables = Auto.columns.drop(['mpg','mpg01','name','acceleration','year'])
design = MS(variables)

X_train = design.fit_transform(Data_train)#.drop(columns=['intercept'])
X_test = design.transform(Data_test)#.drop(columns=['intercept'])
y_train = Data_train['mpg01'].values
y_test = Data_test['mpg01'].values

NB = GaussianNB()
results_NB = NB.fit(X_train,y_train) 

pred_NB = results_NB.predict(X_test)

print("model accuracy error: ",round(float(np.mean(pred_NB != y_test)),3))
confusion_table(pred_NB,y_test)

model accuracy error:  0.076


Truth,0,1
Predicted,,
0,34,4
1,3,51


### KNN

In [ ]:

# Rescale 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)
knn_preds = knn.predict(X_test_scaled)
confusion_table(knn_preds,y_test)

for K in range(1, 20):
    knn  = KNeighborsClassifier(n_neighbors=K)
    pred = knn.fit(X_train_scaled, y_train).predict(X_test_scaled)
    acc  = np.mean(pred != y_test)
    print(f"K={K}: accuracy error {acc:.2%}")

K=1: accuracy error 15.22%
K=2: accuracy error 18.48%
K=3: accuracy error 9.78%
K=4: accuracy error 9.78%
K=5: accuracy error 10.87%
K=6: accuracy error 10.87%
K=7: accuracy error 9.78%
K=8: accuracy error 10.87%
K=9: accuracy error 8.70%
K=10: accuracy error 8.70%
K=11: accuracy error 8.70%
K=12: accuracy error 9.78%
K=13: accuracy error 9.78%
K=14: accuracy error 9.78%
K=15: accuracy error 6.52%
K=16: accuracy error 6.52%
K=17: accuracy error 7.61%
K=18: accuracy error 7.61%
K=19: accuracy error 6.52%
